# Data Cleaning and Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from pathlib import Path

file_name = Path("../data/raw/online_retail_II.xlsx")
if not file_name.is_file():
    raise FileNotFoundError(
        "Download online_retail_II.xlsx from UCI and place it in data/raw/."
    )

df_2009_2010 = pd.read_excel(
    file_name,
    sheet_name="Year 2009-2010",
    engine="openpyxl"
)

df_2010_2011 = pd.read_excel(
    file_name,
    sheet_name="Year 2010-2011",
    engine="openpyxl"
)

In [ ]:
#add a source-year column so we can still identify which sheet each record came from
df_2009_2010 = df_2009_2010.copy()
df_2010_2011 = df_2010_2011.copy()

df_2009_2010["SourceYear"] = "2009-2010"
df_2010_2011["SourceYear"] = "2010-2011"

In [ ]:
#combine them
df_raw = pd.concat(
    [df_2009_2010, df_2010_2011],
    ignore_index=True
)

In [ ]:
print("Combined shape:", df_raw.shape)
print("Total rows:", len(df_raw))
print("Total columns:", len(df_raw.columns))

In [ ]:
#verify the year counts
print(df_raw["SourceYear"].value_counts())

In [ ]:
#check the date range
print("Start date:", df_raw["InvoiceDate"].min())
print("End date:", df_raw["InvoiceDate"].max())

In [ ]:
df_raw.head()

In [ ]:
df_raw.tail()

**Correct the duplicate removal**

In [ ]:
original_cols = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country"
]

In [ ]:
print(
    "Duplicate rows ignoring SourceYear:",
    df_raw.duplicated(
        subset=original_cols
    ).sum()
)

In [ ]:
df_clean = (
    df_raw
    .drop_duplicates(
        subset=original_cols,
        keep="first"
    )
    .copy()
)

In [ ]:
print("Raw rows:", len(df_raw))
print("Clean rows:", len(df_clean))
print("Total duplicate copies removed:",
      len(df_raw) - len(df_clean))

In [ ]:
#verify no duplicates left
print(
    "Remaining exact duplicates:",
    df_clean.duplicated(
        subset=original_cols
    ).sum()
)

**Rebuild the three datasets from corrected df_clean**

*purchases*

In [ ]:
df_purchase = df_clean[
    (df_clean["Customer ID"].notna()) &
    (df_clean["Quantity"] > 0) &
    (df_clean["Price"] >= 0) &
    (~df_clean["Invoice"].astype(str).str.startswith("C"))
].copy()

*Returns/cancellations:*

In [ ]:
df_returns = df_clean[
    (df_clean["Customer ID"].notna()) &
    (df_clean["Quantity"] < 0) &
    (df_clean["Invoice"].astype(str).str.startswith("C"))
].copy()

*Everything else:*

In [ ]:
purchase_idx = df_purchase.index
return_idx = df_returns.index

df_operational = df_clean[
    ~df_clean.index.isin(
        purchase_idx.union(return_idx)
    )
].copy()

In [ ]:
#verify
print("Purchase rows:", len(df_purchase))
print("Return rows:", len(df_returns))
print("Operational/other rows:", len(df_operational))

print(
    "Total:",
    len(df_purchase)
    + len(df_returns)
    + len(df_operational)
)

print("df_clean total:", len(df_clean))

**Rebuild the merchandise purchase dataset**

In [ ]:
special_codes = [
    "POST",
    "DOT",
    "M",
    "AMAZONFEE",
    "BANK CHARGES",
    "B",
    "TEST001",
    "PADS"
]

In [ ]:
#corrected merchandise dataset

df_merchandise = df_purchase[
    ~df_purchase["StockCode"].astype(str).isin(special_codes)
].copy()

In [ ]:
print("Purchase rows before:", len(df_purchase))
print("Merchandise rows after:", len(df_merchandise))
print(
    "Special-code rows removed:",
    len(df_purchase) - len(df_merchandise)
)

In [ ]:
print(
    "Special codes remaining:",
    df_merchandise[
        df_merchandise["StockCode"].astype(str).isin(special_codes)
    ].shape[0]
)

In [ ]:
#recreate TransactionValue because this is a newly rebuilt dataframe:

df_merchandise["TransactionValue"] = (
    df_merchandise["Quantity"] * df_merchandise["Price"]
)

In [ ]:
print(df_merchandise["TransactionValue"].describe())

**Create the corrected order-level dataset**

In [ ]:
#group primarily by Invoice:

df_orders = (
    df_merchandise
    .groupby("Invoice", as_index=False)
    .agg(
        CustomerID=("Customer ID", "first"),
        OrderDate=("InvoiceDate", "min"),
        Country=("Country", "first"),
        SourceYear=("SourceYear", "first"),

        OrderValue=("TransactionValue", "sum"),
        TotalQuantity=("Quantity", "sum"),
        ProductLines=("StockCode", "count"),
        UniqueProducts=("StockCode", "nunique")
    )
)

In [ ]:
OrderDate=("InvoiceDate", "min")

In [ ]:
print("Rows in df_orders:", len(df_orders))
print("Unique invoices:", df_orders["Invoice"].nunique())
print("Unique customers:", df_orders["CustomerID"].nunique())

#Rows in df_orders: 36644 = Unique invoices: 36644

In [ ]:
#every invoice appears once
print(
    "Duplicate invoices in df_orders:",
    df_orders["Invoice"].duplicated().sum()
)

**Inspect the order variables**

In [ ]:
df_orders[
    [
        "OrderValue",
        "TotalQuantity",
        "ProductLines",
        "UniqueProducts"
    ]
].describe()

In [ ]:
df_orders.head(20)

In [ ]:
df_orders.nlargest(
    20,
    "OrderValue"
)[
    [
        "Invoice",
        "CustomerID",
        "OrderDate",
        "OrderValue",
        "TotalQuantity",
        "ProductLines",
        "UniqueProducts",
        "Country"
    ]
]

**Build the customer-level dataset-one row per customer**

In [ ]:
# For each customer, we will initially create:

# Frequency = number of orders
# Monetary = total amount spent
# AverageOrderValue
# TotalQuantity
# AverageBasketQuantity
# AverageUniqueProducts
# FirstPurchaseDate
# LastPurchaseDate

df_customers = (
    df_orders
    .groupby("CustomerID", as_index=False)
    .agg(
        Frequency=("Invoice", "nunique"),
        Monetary=("OrderValue", "sum"),
        AverageOrderValue=("OrderValue", "mean"),
        TotalQuantity=("TotalQuantity", "sum"),
        AverageBasketQuantity=("TotalQuantity", "mean"),
        AverageUniqueProducts=("UniqueProducts", "mean"),
        FirstPurchaseDate=("OrderDate", "min"),
        LastPurchaseDate=("OrderDate", "max")
    )
)

In [ ]:
print("Rows in customer dataset:", len(df_customers))
print("Unique customers:", df_customers["CustomerID"].nunique())

In [ ]:
df_customers.head(20)

In [ ]:
df_customers[
    [
        "Frequency",
        "Monetary",
        "AverageOrderValue",
        "TotalQuantity",
        "AverageBasketQuantity",
        "AverageUniqueProducts"
    ]
].describe()

**Add Recency and Customer Tenure**

In [ ]:
#reference date just after the last transaction date:
#Recency = how many days since the customer's most recent purchase.
#Smaller recency usually means the customer purchased more recently.
#CustomerTenure = how long the customer's observed purchase history spans.

reference_date = df_orders["OrderDate"].max() + pd.Timedelta(days=1)

print("Reference date:", reference_date)

In [ ]:
df_customers["Recency"] = (
    reference_date - df_customers["LastPurchaseDate"]
).dt.days

df_customers["CustomerTenure"] = (
    df_customers["LastPurchaseDate"]
    - df_customers["FirstPurchaseDate"]
).dt.days

In [ ]:
df_customers[
    [
        "CustomerID",
        "FirstPurchaseDate",
        "LastPurchaseDate",
        "Recency",
        "CustomerTenure"
    ]
].head(20)

In [ ]:
df_customers[
    ["Recency", "CustomerTenure"]
].describe()

**Add Product Diversity per Customer**-measure how many different products each customer purchased across the whole dataset.
Customer A repeatedly buys only 2 products
Customer B buys 50 different products

In [ ]:
customer_product_diversity = (
    df_merchandise
    .groupby("Customer ID")["StockCode"]
    .nunique()
    .reset_index()
    .rename(
        columns={
            "Customer ID": "CustomerID",
            "StockCode": "ProductDiversity"
        }
    )
)

In [ ]:
df_customers = df_customers.merge(
    customer_product_diversity,
    on="CustomerID",
    how="left"
)

In [ ]:
df_customers[
    [
        "CustomerID",
        "Frequency",
        "Monetary",
        "ProductDiversity"
    ]
].head(20)

In [ ]:
df_customers["ProductDiversity"].describe()

**Prepare Customer Return Behaviour**

In [ ]:
#remove the same non-merchandise special codes from the return dataset:

df_returns_merchandise = df_returns[
    ~df_returns["StockCode"].astype(str).isin(special_codes)
].copy()

In [ ]:
print("All return rows:", len(df_returns))
print("Merchandise return rows:", len(df_returns_merchandise))
print(
    "Special return rows removed:",
    len(df_returns) - len(df_returns_merchandise)
)

In [ ]:
#return quantities are negative, create a positive quantity representing the number of items returned:

df_returns_merchandise["ReturnedQuantity"] = (
    df_returns_merchandise["Quantity"].abs()
)

In [ ]:
#create the monetary value of the return:

df_returns_merchandise["ReturnValue"] = (
    df_returns_merchandise["ReturnedQuantity"]
    * df_returns_merchandise["Price"]
)

In [ ]:
df_returns_merchandise[
    [
        "Invoice",
        "Customer ID",
        "StockCode",
        "Quantity",
        "ReturnedQuantity",
        "Price",
        "ReturnValue"
    ]
].head(20)

In [ ]:
print(
    df_returns_merchandise[
        ["ReturnedQuantity", "ReturnValue"]
    ].describe()
)

**Create Customer-Level Return Features**

In [ ]:
customer_returns = (
    df_returns_merchandise
    .groupby("Customer ID", as_index=False)
    .agg(
        ReturnTransactions=("Invoice", "nunique"),
        ReturnedQuantity=("ReturnedQuantity", "sum"),
        TotalReturnValue=("ReturnValue", "sum")
    )
    .rename(
        columns={
            "Customer ID": "CustomerID"
        }
    )
)

In [ ]:
df_customers = df_customers.merge(
    customer_returns,
    on="CustomerID",
    how="left"
)

In [ ]:
# A customer without recorded returns has zero returns in these features.

return_columns = [
    "ReturnTransactions",
    "ReturnedQuantity",
    "TotalReturnValue"
]

df_customers[return_columns] = (
    df_customers[return_columns].fillna(0)
)

In [ ]:
df_customers[
    [
        "CustomerID",
        "Frequency",
        "Monetary",
        "ReturnTransactions",
        "ReturnedQuantity",
        "TotalReturnValue"
    ]
].head(20)

In [ ]:
df_customers[
    [
        "ReturnTransactions",
        "ReturnedQuantity",
        "TotalReturnValue"
    ]
].describe()

In [ ]:
#check how many customers made at least one return:
#Monetary = gross merchandise purchases
#TotalReturnValue = merchandise cancellations/returns

customers_with_returns = (
    df_customers["ReturnTransactions"] > 0
).sum()

print("Customers with at least one return:", customers_with_returns)

print(
    "Customers with no returns:",
    (df_customers["ReturnTransactions"] == 0).sum()
)

**Create Return Rate**

In [ ]:
df_customers["ReturnRate"] = (
    df_customers["TotalReturnValue"] /
    df_customers["Monetary"]
)

In [ ]:
df_customers["ReturnRate"] = (
    df_customers["ReturnRate"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)

In [ ]:
df_customers["ReturnRate"].describe()

In [ ]:
df_customers.nlargest(
    20,
    "ReturnRate"
)[
    [
        "CustomerID",
        "Monetary",
        "TotalReturnValue",
        "ReturnTransactions",
        "ReturnRate"
    ]
]

In [ ]:
df_customers.nlargest(
    20,
    "ReturnRate"
)[
    [
        "CustomerID",
        "Monetary",
        "TotalReturnValue",
        "ReturnTransactions",
        "ReturnRate"
    ]
]

**Create Purchase Interval Features**

Measure how regularly customers purchase.

For customers with multiple orders, we can calculate the time gaps between purchases.

In [ ]:
#sort the orders:

df_orders = df_orders.sort_values(
    ["CustomerID", "OrderDate"]
).copy()

In [ ]:
#calculate the gap between consecutive orders:

df_orders["DaysSincePreviousOrder"] = (
    df_orders
    .groupby("CustomerID")["OrderDate"]
    .diff()
    .dt.days
)

In [ ]:
customer_intervals = (
    df_orders
    .groupby("CustomerID")["DaysSincePreviousOrder"]
    .mean()
    .reset_index()
    .rename(
        columns={
            "DaysSincePreviousOrder": "AveragePurchaseInterval"
        }
    )
)

In [ ]:
df_customers = df_customers.merge(
    customer_intervals,
    on="CustomerID",
    how="left"
)

In [ ]:
df_customers[
    [
        "CustomerID",
        "Frequency",
        "AveragePurchaseInterval"
    ]
].head(20)

In [ ]:
df_customers["AveragePurchaseInterval"].describe()

**Create Active Purchase Days**

In [ ]:
df_orders["PurchaseDate"] = (
    df_orders["OrderDate"].dt.date
)

In [ ]:
#calculate unique purchase days for every customer:

active_days = (
    df_orders
    .groupby("CustomerID")["PurchaseDate"]
    .nunique()
    .reset_index()
    .rename(
        columns={
            "PurchaseDate": "ActivePurchaseDays"
        }
    )
)

In [ ]:
df_customers = df_customers.merge(
    active_days,
    on="CustomerID",
    how="left"
)

In [ ]:
df_customers["OrdersPerActiveDay"] = (
    df_customers["Frequency"] /
    df_customers["ActivePurchaseDays"]
)

In [ ]:
df_customers[
    [
        "CustomerID",
        "Frequency",
        "ActivePurchaseDays",
        "OrdersPerActiveDay"
    ]
].head(20)

In [ ]:
df_customers[
    [
        "ActivePurchaseDays",
        "OrdersPerActiveDay"
    ]
].describe()

**Final quality check of the customer-level dataset**

In [ ]:
print("Rows:", df_customers.shape[0])
print("Columns:", df_customers.shape[1])
print("Unique customers:", df_customers["CustomerID"].nunique())

In [ ]:
print(
    "Duplicate Customer IDs:",
    df_customers["CustomerID"].duplicated().sum()
)

In [ ]:
df_customers.isnull().sum().sort_values(ascending=False)

In [ ]:
#Check infinite values
numeric_cols = df_customers.select_dtypes(include=np.number).columns

print(
    "Total infinite values:",
    np.isinf(df_customers[numeric_cols]).sum().sum()
)

In [ ]:
print(df_customers.columns.tolist())

Log Transformation of Skewed Customer Features

Customer behavioural variables are highly right-skewed due to a small number
of customers with very high purchasing activity.

Log transformation is applied to suitable non-negative numerical variables
before parametric statistical inference and modelling.

Original variables are retained for interpretation.

In [ ]:
#Check skewness before transforming



skew_check_cols = [
    "Frequency",
    "Monetary",
    "AverageOrderValue",
    "TotalQuantity",
    "AverageBasketQuantity",
    "AverageUniqueProducts",
    "Recency",
    "CustomerTenure",
    "ProductDiversity",
    "ReturnTransactions",
    "ReturnedQuantity",
    "TotalReturnValue",
    "ReturnRate",
    "AveragePurchaseInterval",
    "ActivePurchaseDays",
    "OrdersPerActiveDay"
]

skew_before = (
    df_customers[skew_check_cols]
    .skew()
    .sort_values(ascending=False)
)

print("Skewness before transformation:")
print(skew_before)

In [ ]:
# Features selected for log transformation
log_transform_cols = [
    "TotalReturnValue",
    "AverageOrderValue",
    "AverageBasketQuantity",
    "ReturnedQuantity",
    "Monetary",
    "TotalQuantity",
    "Frequency",
    "ReturnRate",
    "ReturnTransactions",
    "ActivePurchaseDays",
    "OrdersPerActiveDay",
    "ProductDiversity",
    "AverageUniqueProducts"
]

# Create new log-transformed columns
# Keep the original columns unchanged
for col in log_transform_cols:
    df_customers["Log_" + col] = np.log1p(df_customers[col])

In [ ]:
log_cols = ["Log_" + col for col in log_transform_cols]

skew_after = (
    df_customers[log_cols]
    .skew()
    .sort_values(ascending=False)
)

print("Skewness after log transformation:")
print(skew_after)

In [ ]:
# AverageOrderValue       ← original, business interpretation
# Log_AverageOrderValue   ← statistical analysis when needed



log_cols = ["Log_" + col for col in log_transform_cols]

print("Infinite values:",
      np.isinf(df_customers[log_cols]).sum().sum())

print("\nMissing values:")
print(df_customers[log_cols].isnull().sum())

In [ ]:
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

df_clean.to_csv(
    output_dir / "online_retail_clean.csv",
    index=False
)

df_merchandise.to_csv(
    output_dir / "online_retail_merchandise.csv",
    index=False
)

df_orders.to_csv(
    output_dir / "order_level_data.csv",
    index=False
)

df_customers.to_csv(
    output_dir / "customer_level_data.csv",
    index=False
)

In [ ]:
print("Saved successfully!")

print("Clean transactions:", df_clean.shape)
print("Merchandise transactions:", df_merchandise.shape)
print("Orders:", df_orders.shape)
print("Customers:", df_customers.shape)